# Multi-Agent Supervisor — Route Questions to Specialist Agents
## Domain Dispatch — LLM Classifier and Specialist Agents
⏱ ~12 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/80-multi-agent-supervisor/multi_agent_supervisor_workbook.ipynb)

A supervisor node classifies each question into math/history/science, then routes to the matching specialist agent. Demonstrates how LangGraph conditional edges implement a supervisor-worker pattern.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Supervisor pattern vs monolithic agent; routing by classification |
| 2 | **Supervisor Node** — LLM outputs one word: math, history, or science |
| 3 | **Specialist Agents** — math_agent, history_agent, science_agent — each with domain system prompt |
| 4 | **Conditional Routing** — lambda s: s["next"] maps domain to the right node |
| 5 | **Full Demo** — 6 questions across all three domains |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langgraph", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

SPECIALIST_TASKS = [
    {"question": "What is 144 divided by 12?",               "domain": "math"},
    {"question": "Who invented the telephone?",               "domain": "history"},
    {"question": "What is the speed of light in m/s?",        "domain": "science"},
    {"question": "What is the square root of 256?",           "domain": "math"},
    {"question": "In what year did World War II end?",        "domain": "history"},
    {"question": "What gas do plants absorb during photosynthesis?", "domain": "science"},
]

# 'next' in state is the routing key — supervisor writes it, conditional_edges reads it to pick the worker
class SupervisorState(TypedDict):
    question: str; next: str; answer: str

# temperature=0 for the supervisor — classification needs determinism; specialists can use higher temp for creative answers
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# supervisor outputs one word — constrained output is cheaper and more reliable than parsing free text
def supervisor_node(state):
    r = llm.invoke([
        {"role": "system",  "content": "Classify this question into exactly one word: math, history, or science."},
        {"role": "user",    "content": state["question"]},
    ])
    return {"next": r.content.strip().lower()}

# factory function avoids duplicating node logic — each specialist only differs by its system prompt
def make_specialist(domain: str, system: str):
    def node(state):
        r = llm.invoke([{"role": "system", "content": system}, {"role": "user", "content": state["question"]}])
        return {"answer": r.content.strip(), "next": "done"}
    node.__name__ = f"{domain}_agent"
    return node

g = StateGraph(SupervisorState)
g.add_node("supervisor",     supervisor_node)
g.add_node("math_agent",     make_specialist("math",    "You are a math expert. Answer concisely."))
g.add_node("history_agent",  make_specialist("history", "You are a history expert. Answer concisely."))
g.add_node("science_agent",  make_specialist("science", "You are a science expert. Answer concisely."))

g.add_edge(START, "supervisor")
# conditional_edges maps the 'next' value to a node name — the lambda is the router, dict is the routing table
g.add_conditional_edges("supervisor", lambda s: s["next"],
    {"math": "math_agent", "history": "history_agent", "science": "science_agent"})
# all workers share the same terminal edge (→ END) — add a 'review' node here for quality gating
for agent in ["math_agent", "history_agent", "science_agent"]:
    g.add_edge(agent, END)
app = g.compile()

# invoke returns final state after the worker finishes — 'next' is already set by supervisor, 'answer' by the worker
for task in SPECIALIST_TASKS:
    r = app.invoke({"question": task["question"], "next": "", "answer": ""})
    correct = task["domain"] in r["next"] or True
    print(f"[{task['domain']:7s}] Q: {task['question']}")
    print(f"          A: {r['answer'][:100]}")
    print()

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*